# Phase 2 — E5: Swap encoder to CLIP ViT-B/16

CLIP was pretrained on 400M image-text pairs and produces image features
that are aligned to natural-language descriptions — strictly better
caption-relevant features than ImageNet-classification EfficientNet.

Step 1 extracts the 196 patch-level features (14×14 grid, 768-dim) once
over every unique image and caches them. Step 2 trains the standard
Transformer decoder on those cached features, using the same code path as
E0/E4/E7 — only `encoder_feat_dim` and `num_spatial_tokens` change.

In [1]:
import sys
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("output").resolve()))
import importlib
import exp_runner
importlib.reload(exp_runner)
from exp_runner import ExperimentConfig, _get_image_reader, run_experiment

/home/coezbek/dev/2026/AT3_training/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CACHE = Path("output/clip_vitb16_features.pt")

if not CACHE.exists():
    from transformers import CLIPModel, CLIPImageProcessor

    model_id = "openai/clip-vit-base-patch16"
    clip = CLIPModel.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda").eval()
    proc = CLIPImageProcessor.from_pretrained(model_id)

    cfg_tmp = ExperimentConfig(run_name="clip_extract", use_cached_features=False)
    reader = _get_image_reader(cfg_tmp)
    df = pd.read_csv("output/processed_captions.csv")
    uniq = df[["image_id", "file_name"]].drop_duplicates().sort_values("image_id").reset_index(drop=True)
    N = len(uniq)
    D = clip.config.vision_config.hidden_size  # 768
    S = 196  # 14x14 patches after dropping CLS

    feats = torch.empty((N, S, D), dtype=torch.float16)
    image_to_idx: dict[int, int] = {}
    BATCH = 16

    batch_imgs, batch_pos = [], []
    with torch.no_grad():
        for i, row in tqdm(uniq.iterrows(), total=N, desc="CLIP extract"):
            img = reader.read(row["file_name"])
            batch_imgs.append(img)
            batch_pos.append(i)
            image_to_idx[int(row["image_id"])] = i
            if len(batch_imgs) == BATCH:
                px = proc(images=batch_imgs, return_tensors="pt")["pixel_values"].to("cuda", dtype=torch.float16)
                out = clip.vision_model(pixel_values=px).last_hidden_state
                out = out[:, 1:, :].cpu().to(torch.float16)  # drop CLS
                for k, p in enumerate(batch_pos):
                    feats[p] = out[k]
                batch_imgs, batch_pos = [], []
        if batch_imgs:
            px = proc(images=batch_imgs, return_tensors="pt")["pixel_values"].to("cuda", dtype=torch.float16)
            out = clip.vision_model(pixel_values=px).last_hidden_state
            out = out[:, 1:, :].cpu().to(torch.float16)
            for k, p in enumerate(batch_pos):
                feats[p] = out[k]

    torch.save({"features": feats, "image_to_idx": image_to_idx}, CACHE)
    print(f"Saved CLIP cache {tuple(feats.shape)} to {CACHE}")

    del clip
    import gc
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"CLIP cache already at {CACHE}")

(null): No such file or directory


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 19699.70it/s]

CLIP extract:   0%|          | 0/7750 [00:00<?, ?it/s]

CLIP extract:   0%|          | 10/7750 [00:00<01:22, 93.27it/s]

...

CLIP extract: 100%|█████████▉| 7712/7750 [01:41<00:00, 77.90it/s]

CLIP extract: 100%|█████████▉| 7728/7750 [01:41<00:00, 80.45it/s]

CLIP extract: 100%|█████████▉| 7744/7750 [01:41<00:00, 79.28it/s]

CLIP extract: 100%|██████████| 7750/7750 [01:41<00:00, 76.08it/s]

Saved CLIP cache (7750, 196, 768) to output/clip_vitb16_features.pt


In [3]:
cfg = ExperimentConfig(
    run_name="phase2_e5_clip",
    output_dir="output/phase2_results",
    feature_cache=str(CACHE),
    encoder_kind="efficientnet_b0_cached",  # generic cached-feature head
    encoder_feat_dim=768,
    num_spatial_tokens=196,
    epochs=15,
    batch_size=128,
    lr=1e-4,
    decoding="beam",
    beam_width=5,
    length_penalty=0.7,
)
metrics_e5 = run_experiment(cfg)
metrics_e5

[phase2_e5_clip] device=cuda


[phase2_e5_clip] starting training: epochs=15 batches/epoch=207


[phase2_e5_clip] ep01  batch 41/207  train_loss(running)=6.0613  elapsed=15.1s


[phase2_e5_clip] ep01  batch 82/207  train_loss(running)=5.3495  elapsed=30.4s


[phase2_e5_clip] ep01  batch 123/207  train_loss(running)=5.0317  elapsed=45.7s


...

[phase2_e5_clip] training finished after 15 epochs; starting eval (beam).


[phase2_e5_clip] val eval done — BLEU-4=0.3840  CIDEr=1.4124


[phase2_e5_clip] test eval done — BLEU-4=0.3890  CIDEr=1.4744


[phase2_e5_clip] DONE  val_BLEU-4=0.3840  test_BLEU-4=0.3890  test_CIDEr=1.4744


{'run_name': 'phase2_e5_clip',
 'best_val_loss': 2.9365400950113933,
 'epochs_run': 15,
 'decoding': 'beam',
 'beam_width': 5,
 'length_penalty': 0.7,
 'val_BLEU-1': 0.6377301373255732,
 'val_BLEU-2': 0.5237676807591981,
 'val_BLEU-3': 0.44227570086111156,
 'val_BLEU-4': 0.3839519486770217,
 'val_CIDEr': 1.4124239992494292,
 'test_BLEU-1': 0.6315817257068856,
 'test_BLEU-2': 0.5215425828168714,
 'test_BLEU-3': 0.44392812333644227,
 'test_BLEU-4': 0.38897140447975675,
 'test_CIDEr': 1.4744476432605012}